# Phase 2 — BGP Time-Series Analysis

This notebook is part of the **BGP AI Analytics Engine** project.

The purpose of this notebook is to analyze the dynamic BGP event dataset captured from the **Route Views BMP streaming service**.

Unlike the static BGP dataset analyzed in earlier phases, this dataset contains BGP events observed over time, allowing the project to move from static route analysis toward **dynamic BGP time-series analytics**.

## Objective

The main objectives of this notebook are to:

- Validate the captured BGP event dataset
- Understand the structure and quality of the time-series data
- Analyze BGP event distribution over time
- Analyze Announcement and Withdrawal events
- Identify prefixes with unusually high numbers of BGP events
- Investigate repeated Announcement / Withdrawal patterns
- Establish the foundation for detecting possible route instability and flapping

## BGP Event Model

Each record represents a BGP event observed in the streaming data.

The main event types are:

- **A — Announcement**
- **W — Withdrawal**

The timestamp associated with each event allows the same prefix to be observed as its routing state changes over time.

For example:

    Timestamp    Prefix          Event
    ----------   -------------   -----
    01:51:02     203.0.113.0/24  A
    01:51:15     203.0.113.0/24  W
    01:51:22     203.0.113.0/24  A
    01:51:39     203.0.113.0/24  W

Repeated Announcement / Withdrawal activity may indicate **route instability or route flapping**.

However, repeated events alone do not prove a routing loop. Further analysis and correlation with peer, AS path, next-hop, and timing information are required to investigate the possible cause.

## Analysis Flow

Raw BGP Event Dataset

↓

Data Validation

↓

Time-Series Profiling

↓

Event Distribution Analysis

↓

Prefix / Peer Analysis

↓

Announcement / Withdrawal Pattern Analysis

↓

Route Instability Indicators

↓

Future Feature Engineering and AI/ML

## Data Source

The dataset analyzed in this notebook is generated by:

**Route Views BMP Stream → pybgpstream → BGP Events → CSV**

The raw dataset is stored under:

`data/raw_data/`

## Scope

This notebook focuses on **understanding and analyzing the captured BGP events**.

AI/ML-based anomaly detection and network intelligence will be developed in later stages after the characteristics of the BGP event data and suitable features have been established.

In [ ]:
## Analysis Philosophy

This phase focuses on understanding BGP behavior using data analytics and network engineering knowledge.

The objective is to identify patterns, trends, anomalies, and operational insights from the BGP event dataset before introducing machine learning techniques.

Network engineering expertise remains the primary tool for interpreting the results.

AI and ML will only be considered after the behavior of the data is sufficiently understood and meaningful features have been established.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# Load BGP event dataset
df = pd.read_csv(
    "../data/raw_data/bgp_updates_20260820_014857_UTC.csv"
)

# Convert timestamp
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Count BGP events per minute
events_per_minute = (
    df.set_index("timestamp")
      .resample("1min")
      .size()
)

# Plot
plt.figure(figsize=(12, 5))

plt.plot(
    events_per_minute.index,
    events_per_minute.values
)

plt.title("BGP Events per Minute")
plt.xlabel("Time")
plt.ylabel("Number of BGP Events")

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

Matplotlib is building the font cache; this may take a moment.


FileNotFoundError: [Errno 2] No such file or directory: '../data/raw_data/bgp_updates_20260820_014857_UTC.csv'

In [ ]:
announcements = (
    df[df["type"] == "A"]
    .set_index("timestamp")
    .resample("1min")
    .size()
)

withdrawals = (
    df[df["type"] == "W"]
    .set_index("timestamp")
    .resample("1min")
    .size()
)